# Fase 7: Validación Integral del Flujo de Datos

Auditoría cruzada entre las capas Raw, Bronze, Silver y Gold para comprobar:

- Paridad de registros.
- Conciliación financiera.
- Unicidad de dimensiones.
- Integridad referencial.
- Cobertura temporal.

### 1 - Configuración de rutas y verificación de capas

In [1]:
from pathlib import Path
import pandas as pd

In [3]:
# Detectar la raíz del proyecto aunque el notebook se ejecute desde /notebooks
directorio_actual = Path.cwd()
ruta_base = (
    directorio_actual.parent
    if directorio_actual.name == "notebooks"
    else directorio_actual
)

ruta_raw = ruta_base / "data" / "raw" / "northwind"
ruta_bronze = ruta_base / "data" / "bronze"
ruta_silver = ruta_base / "data" / "silver"
ruta_gold = ruta_base / "data" / "gold"

rutas_capas = {
    "Raw": ruta_raw,
    "Bronze": ruta_bronze,
    "Silver": ruta_silver,
    "Gold": ruta_gold,
}

print("=== VERIFICACIÓN DE CAPAS DEL PIPELINE ===")
print()

todas_disponibles = True

for nombre_capa, ruta_capa in rutas_capas.items():
    existe = ruta_capa.exists()
    estado = "[OK] Disponible" if existe else "[ERROR] No encontrada"
    todas_disponibles = todas_disponibles and existe

    extension = "*.csv" if nombre_capa == "Raw" else "*.parquet"
    cantidad_archivos = (
        len(list(ruta_capa.glob(extension)))
        if existe
        else 0
    )

    print(
        f"{estado:<17} | "
        f"{nombre_capa:<6} | "
        f"{cantidad_archivos:>2} archivos | "
        f"{ruta_capa.resolve()}"
    )

print()

if not todas_disponibles:
    raise FileNotFoundError(
        "Falta una o más capas del pipeline. Revisa las rutas antes de continuar."
    )

print("[OK] Las cuatro capas están disponibles para la auditoría.")

=== VERIFICACIÓN DE CAPAS DEL PIPELINE ===

[OK] Disponible      | Raw    | 11 archivos | data\raw\northwind
[OK] Disponible      | Bronze | 11 archivos | data\bronze
[OK] Disponible      | Silver | 11 archivos | data\silver
[OK] Disponible      | Gold   |  6 archivos | data\gold

[OK] Las cuatro capas están disponibles para la auditoría.


## 2. Conciliación de registros entre capas

Comparación del número de pedidos y líneas de pedido desde Raw hasta Gold, con el fin de detectar pérdidas o duplicaciones durante el pipeline.

In [4]:
### 2 - Conciliación de pedidos y líneas de pedido

# Cargar tablas transaccionales de cada capa
raw_orders = pd.read_csv(ruta_raw / "orders.csv")
raw_details = pd.read_csv(ruta_raw / "order-details.csv")

bronze_orders = pd.read_parquet(ruta_bronze / "orders.parquet")
bronze_details = pd.read_parquet(ruta_bronze / "order-details.parquet")

silver_orders = pd.read_parquet(ruta_silver / "orders.parquet")
silver_details = pd.read_parquet(ruta_silver / "order-details.parquet")

gold_fact = pd.read_parquet(ruta_gold / "Fact_Ventas.parquet")

# Construir tabla comparativa
conciliacion_registros = pd.DataFrame([
    {
        "capa": "Raw",
        "pedidos": raw_orders["orderID"].nunique(),
        "lineasPedido": len(raw_details),
    },
    {
        "capa": "Bronze",
        "pedidos": bronze_orders["orderID"].nunique(),
        "lineasPedido": len(bronze_details),
    },
    {
        "capa": "Silver",
        "pedidos": silver_orders["orderID"].nunique(),
        "lineasPedido": len(silver_details),
    },
    {
        "capa": "Gold",
        "pedidos": gold_fact["orderID"].nunique(),
        "lineasPedido": len(gold_fact),
    },
])

In [5]:
# Calcular diferencias respecto de la fuente Raw
pedidos_raw = conciliacion_registros.loc[
    conciliacion_registros["capa"] == "Raw", "pedidos"
].iloc[0]

lineas_raw = conciliacion_registros.loc[
    conciliacion_registros["capa"] == "Raw", "lineasPedido"
].iloc[0]

conciliacion_registros["diferenciaPedidos"] = (
    conciliacion_registros["pedidos"] - pedidos_raw
)

conciliacion_registros["diferenciaLineas"] = (
    conciliacion_registros["lineasPedido"] - lineas_raw
)

print("=== CONCILIACIÓN DE REGISTROS ENTRE CAPAS ===")
print()
print(conciliacion_registros.to_string(index=False))
print()



=== CONCILIACIÓN DE REGISTROS ENTRE CAPAS ===

  capa  pedidos  lineasPedido  diferenciaPedidos  diferenciaLineas
   Raw      830          2155                  0                 0
Bronze      830          2155                  0                 0
Silver      830          2155                  0                 0
  Gold      830          2155                  0                 0



In [6]:
registros_ok = (
    conciliacion_registros["diferenciaPedidos"].eq(0).all()
    and conciliacion_registros["diferenciaLineas"].eq(0).all()
)

if not registros_ok:
    raise ValueError(
        "La conciliación falló: existen diferencias de registros entre capas."
    )

print("[OK] Conciliación correcta: no se perdieron ni duplicaron registros.")

[OK] Conciliación correcta: no se perdieron ni duplicaron registros.


## 3. Conciliación financiera entre capas

Recálculo independiente de ventas brutas, descuentos y ventas netas en cada capa, aplicando redondeo monetario por línea.

In [7]:
### 3 - Conciliación financiera entre Raw, Bronze, Silver y Gold

def calcular_totales_financieros(df):
    """Calcula métricas monetarias aplicando redondeo por línea."""
    importe_bruto = (
        df["quantity"] * df["unitPrice"]
    ).round(2)

    importe_descuento = (
        importe_bruto * df["discount"]
    ).round(2)

    importe_neto = (
        importe_bruto - importe_descuento
    ).round(2)

    return {
        "ventaBruta": round(importe_bruto.sum(), 2),
        "descuentos": round(importe_descuento.sum(), 2),
        "ventaNeta": round(importe_neto.sum(), 2),
    }

In [8]:
# Calcular totales independientemente en cada capa
totales_por_capa = {
    "Raw": calcular_totales_financieros(raw_details),
    "Bronze": calcular_totales_financieros(bronze_details),
    "Silver": calcular_totales_financieros(silver_details),
    "Gold": calcular_totales_financieros(gold_fact),
}

conciliacion_financiera = pd.DataFrame.from_dict(
    totales_por_capa,
    orient="index"
).reset_index(names="capa")

In [9]:
# Usar Raw como valor de referencia
totales_raw = conciliacion_financiera.loc[
    conciliacion_financiera["capa"] == "Raw"
].iloc[0]

conciliacion_financiera["difVentaBruta"] = (
    conciliacion_financiera["ventaBruta"] - totales_raw["ventaBruta"]
).round(2)

conciliacion_financiera["difDescuentos"] = (
    conciliacion_financiera["descuentos"] - totales_raw["descuentos"]
).round(2)

conciliacion_financiera["difVentaNeta"] = (
    conciliacion_financiera["ventaNeta"] - totales_raw["ventaNeta"]
).round(2)

print("=== CONCILIACIÓN FINANCIERA ENTRE CAPAS ===")
print()

print(
    conciliacion_financiera.to_string(
        index=False,
        formatters={
            "ventaBruta": "${:,.2f}".format,
            "descuentos": "${:,.2f}".format,
            "ventaNeta": "${:,.2f}".format,
            "difVentaBruta": "${:,.2f}".format,
            "difDescuentos": "${:,.2f}".format,
            "difVentaNeta": "${:,.2f}".format,
        }
    )
)

print()

columnas_diferencias = [
    "difVentaBruta",
    "difDescuentos",
    "difVentaNeta",
]

finanzas_ok = (
    conciliacion_financiera[columnas_diferencias]
    .eq(0)
    .all()
    .all()
)

if not finanzas_ok:
    raise ValueError(
        "La conciliación financiera falló: existen diferencias monetarias."
    )

print("[OK] Conciliación financiera exacta: diferencia de $0.00 en todas las capas.")

=== CONCILIACIÓN FINANCIERA ENTRE CAPAS ===

  capa    ventaBruta descuentos     ventaNeta difVentaBruta difDescuentos difVentaNeta
   Raw $1,354,458.59 $88,665.57 $1,265,793.02         $0.00         $0.00        $0.00
Bronze $1,354,458.59 $88,665.57 $1,265,793.02         $0.00         $0.00        $0.00
Silver $1,354,458.59 $88,665.57 $1,265,793.02         $0.00         $0.00        $0.00
  Gold $1,354,458.59 $88,665.57 $1,265,793.02         $0.00         $0.00        $0.00

[OK] Conciliación financiera exacta: diferencia de $0.00 en todas las capas.


## 4. Validación de unicidad de las dimensiones Gold

Comprobación de claves primarias únicas y no nulas para garantizar relaciones de cardinalidad 1:N con `Fact_Ventas`.

In [16]:
### 4 - Validación de unicidad y claves nulas en dimensiones Gold

dimensiones_gold = {
    "Dim_Calendario": {
        "dataframe": pd.read_parquet(
            ruta_gold / "Dim_Calendario.parquet"
        ),
        "clave": "date",
    },
    "Dim_Cliente": {
        "dataframe": pd.read_parquet(
            ruta_gold / "Dim_Cliente.parquet"
        ),
        "clave": "customerID",
    },
    "Dim_Producto": {
        "dataframe": pd.read_parquet(
            ruta_gold / "Dim_Producto.parquet"
        ),
        "clave": "productID",
    },
    "Dim_Empleado": {
        "dataframe": pd.read_parquet(
            ruta_gold / "Dim_Empleado.parquet"
        ),
        "clave": "employeeID",
    },
    "Dim_Transportista": {
        "dataframe": pd.read_parquet(
            ruta_gold / "Dim_Transportista.parquet"
        ),
        "clave": "shipperID",
    },
}


In [11]:
resultados_dimensiones = []

for nombre_dimension, configuracion in dimensiones_gold.items():
    df_dimension = configuracion["dataframe"]
    clave = configuracion["clave"]

    filas = len(df_dimension)
    valores_unicos = df_dimension[clave].nunique(dropna=True)
    claves_duplicadas = df_dimension[clave].duplicated().sum()
    claves_nulas = df_dimension[clave].isna().sum()

    estado = (
        "[OK]"
        if claves_duplicadas == 0 and claves_nulas == 0
        else "[ERROR]"
    )

    resultados_dimensiones.append({
        "dimension": nombre_dimension,
        "clavePrimaria": clave,
        "filas": filas,
        "valoresUnicos": valores_unicos,
        "duplicados": claves_duplicadas,
        "nulos": claves_nulas,
        "estado": estado,
    })

validacion_dimensiones = pd.DataFrame(resultados_dimensiones)

print("=== VALIDACIÓN DE DIMENSIONES GOLD ===")
print()
print(validacion_dimensiones.to_string(index=False))
print()

=== VALIDACIÓN DE DIMENSIONES GOLD ===

        dimension clavePrimaria  filas  valoresUnicos  duplicados  nulos estado
   Dim_Calendario          date   1096           1096           0      0   [OK]
      Dim_Cliente    customerID     91             91           0      0   [OK]
     Dim_Producto     productID     77             77           0      0   [OK]
     Dim_Empleado    employeeID      9              9           0      0   [OK]
Dim_Transportista     shipperID      3              3           0      0   [OK]



In [12]:
dimensiones_ok = (
    validacion_dimensiones["duplicados"].eq(0).all()
    and validacion_dimensiones["nulos"].eq(0).all()
    and (
        validacion_dimensiones["filas"]
        == validacion_dimensiones["valoresUnicos"]
    ).all()
)

if not dimensiones_ok:
    raise ValueError(
        "Existen claves duplicadas o nulas en las dimensiones Gold."
    )

print("[OK] Todas las dimensiones tienen claves primarias únicas y no nulas.")
print("[OK] La cardinalidad 1:N del modelo estrella está garantizada.")

[OK] Todas las dimensiones tienen claves primarias únicas y no nulas.
[OK] La cardinalidad 1:N del modelo estrella está garantizada.


## 5. Validación de cobertura temporal

Verificación de que `Dim_Calendario` sea continua y cubra todas las fechas de pedido registradas en `Fact_Ventas`.

In [13]:
### 5 - Validación de cobertura temporal

dim_calendario = dimensiones_gold["Dim_Calendario"]["dataframe"].copy()

fechas_calendario = (
    pd.to_datetime(dim_calendario["date"])
    .dt.normalize()
    .sort_values()
    .reset_index(drop=True)
)

fechas_ventas = (
    pd.to_datetime(gold_fact["orderDate"])
    .dt.normalize()
)

In [14]:
# Verificar fechas nulas o fuera del calendario
fechas_nulas = fechas_ventas.isna().sum()
fechas_fuera_calendario = (~fechas_ventas.isin(fechas_calendario)).sum()

# Verificar continuidad diaria de la dimensión
diferencias_dias = fechas_calendario.diff().dropna().dt.days
saltos_calendario = diferencias_dias.ne(1).sum()


In [15]:
# Resumen temporal
print("=== VALIDACIÓN DE COBERTURA TEMPORAL ===")
print()
print(
    f"Rango Dim_Calendario : "
    f"{fechas_calendario.min().date()} a "
    f"{fechas_calendario.max().date()}"
)
print(
    f"Rango de ventas       : "
    f"{fechas_ventas.min().date()} a "
    f"{fechas_ventas.max().date()}"
)
print(f"Días en calendario    : {len(fechas_calendario)}")
print(f"Saltos en calendario  : {saltos_calendario}")
print(f"Fechas de venta nulas : {fechas_nulas}")
print(f"Ventas fuera de rango : {fechas_fuera_calendario}")
print()

cobertura_temporal_ok = (
    saltos_calendario == 0
    and fechas_nulas == 0
    and fechas_fuera_calendario == 0
)

if not cobertura_temporal_ok:
    raise ValueError(
        "La dimensión calendario no cubre correctamente las fechas de venta."
    )

print("[OK] Dim_Calendario contiene una secuencia diaria continua.")
print("[OK] Todas las ventas están cubiertas por la dimensión temporal.")

=== VALIDACIÓN DE COBERTURA TEMPORAL ===

Rango Dim_Calendario : 1996-01-01 a 1998-12-31
Rango de ventas       : 1996-07-04 a 1998-05-06
Días en calendario    : 1096
Saltos en calendario  : 0
Fechas de venta nulas : 0
Ventas fuera de rango : 0

[OK] Dim_Calendario contiene una secuencia diaria continua.
[OK] Todas las ventas están cubiertas por la dimensión temporal.


## 6. Validación de integridad referencial en Gold

Comprobación de que todas las claves foráneas de `Fact_Ventas` existan en sus dimensiones correspondientes.

In [17]:
### 6 - Validación de integridad referencial del modelo Gold

relaciones_gold = [
    {
        "claveHecho": "productID",
        "dimension": "Dim_Producto",
        "claveDimension": "productID",
    },
    {
        "claveHecho": "customerID",
        "dimension": "Dim_Cliente",
        "claveDimension": "customerID",
    },
    {
        "claveHecho": "employeeID",
        "dimension": "Dim_Empleado",
        "claveDimension": "employeeID",
    },
    {
        "claveHecho": "shipperID",
        "dimension": "Dim_Transportista",
        "claveDimension": "shipperID",
    },
]

resultados_integridad = []

for relacion in relaciones_gold:
    clave_hecho = relacion["claveHecho"]
    nombre_dimension = relacion["dimension"]
    clave_dimension = relacion["claveDimension"]

    df_dimension = dimensiones_gold[nombre_dimension]["dataframe"]

    valores_nulos = gold_fact[clave_hecho].isna().sum()
    claves_huerfanas = (
        ~gold_fact[clave_hecho].isin(df_dimension[clave_dimension])
    ).sum()

    estado = (
        "[OK]"
        if valores_nulos == 0 and claves_huerfanas == 0
        else "[ERROR]"
    )

    resultados_integridad.append({
        "claveHecho": clave_hecho,
        "dimension": nombre_dimension,
        "nulos": valores_nulos,
        "huerfanas": claves_huerfanas,
        "estado": estado,
    })

validacion_integridad = pd.DataFrame(resultados_integridad)

print("=== VALIDACIÓN DE INTEGRIDAD REFERENCIAL ===")
print()
print(validacion_integridad.to_string(index=False))
print()

integridad_referencial_ok = (
    validacion_integridad["nulos"].eq(0).all()
    and validacion_integridad["huerfanas"].eq(0).all()
    and fechas_nulas == 0
    and fechas_fuera_calendario == 0
)

if not integridad_referencial_ok:
    raise ValueError(
        "La tabla Fact_Ventas contiene claves nulas o huérfanas."
    )

print("[OK] Todas las claves foráneas tienen correspondencia en sus dimensiones.")
print("[OK] La integridad referencial del modelo Gold está garantizada.")

=== VALIDACIÓN DE INTEGRIDAD REFERENCIAL ===

claveHecho         dimension  nulos  huerfanas estado
 productID      Dim_Producto      0          0   [OK]
customerID       Dim_Cliente      0          0   [OK]
employeeID      Dim_Empleado      0          0   [OK]
 shipperID Dim_Transportista      0          0   [OK]

[OK] Todas las claves foráneas tienen correspondencia en sus dimensiones.
[OK] La integridad referencial del modelo Gold está garantizada.


## 7. Resultado consolidado de la auditoría

Resumen ejecutivo de las pruebas realizadas sobre el pipeline completo y certificación del modelo analítico.

In [18]:
### 7 - Consolidación de resultados de la auditoría

controles_auditoria = [
    {
        "control": "Disponibilidad de las cuatro capas",
        "resultado": bool(todas_disponibles),
    },
    {
        "control": "Conciliación de pedidos y líneas",
        "resultado": bool(registros_ok),
    },
    {
        "control": "Conciliación financiera",
        "resultado": bool(finanzas_ok),
    },
    {
        "control": "Unicidad de dimensiones Gold",
        "resultado": bool(dimensiones_ok),
    },
    {
        "control": "Cobertura temporal",
        "resultado": bool(cobertura_temporal_ok),
    },
    {
        "control": "Integridad referencial",
        "resultado": bool(integridad_referencial_ok),
    },
]

resumen_auditoria = pd.DataFrame(controles_auditoria)

resumen_auditoria["estado"] = resumen_auditoria["resultado"].map({
    True: "[OK]",
    False: "[ERROR]",
})

print("=== RESULTADO CONSOLIDADO DE LA AUDITORÍA ===")
print()
print(
    resumen_auditoria[
        ["control", "estado"]
    ].to_string(index=False)
)
print()

auditoria_aprobada = resumen_auditoria["resultado"].all()

if not auditoria_aprobada:
    controles_fallidos = resumen_auditoria.loc[
        ~resumen_auditoria["resultado"],
        "control",
    ].tolist()

    raise ValueError(
        f"Auditoría rechazada. Controles fallidos: {controles_fallidos}"
    )

print("=" * 70)
print("[OK] AUDITORÍA APROBADA")
print("[OK] El modelo Gold representa fielmente los datos de origen.")
print("[OK] Diferencia financiera entre Raw y Gold: $0.00")
print("[OK] El pipeline está listo para su integración con Power BI.")
print("=" * 70)

=== RESULTADO CONSOLIDADO DE LA AUDITORÍA ===

                           control estado
Disponibilidad de las cuatro capas   [OK]
  Conciliación de pedidos y líneas   [OK]
           Conciliación financiera   [OK]
      Unicidad de dimensiones Gold   [OK]
                Cobertura temporal   [OK]
            Integridad referencial   [OK]

[OK] AUDITORÍA APROBADA
[OK] El modelo Gold representa fielmente los datos de origen.
[OK] Diferencia financiera entre Raw y Gold: $0.00
[OK] El pipeline está listo para su integración con Power BI.


## 8. Generación del acta técnica de validación

Exportación de los resultados consolidados de la auditoría a un documento Markdown dentro de `docs/`.

In [19]:
### 8 - Generar acta técnica de validación

from datetime import datetime

ruta_docs = ruta_base / "docs"
ruta_docs.mkdir(parents=True, exist_ok=True)

ruta_acta = ruta_docs / "acta_validacion_pipeline.md"

fecha_validacion = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

tabla_controles = resumen_auditoria[
    ["control", "estado"]
].to_markdown(index=False)

contenido_acta = f"""# Acta de Validación Integral del Pipeline Northwind BI

## Información general

- **Fecha de validación:** {fecha_validacion}
- **Capas auditadas:** Raw, Bronze, Silver y Gold
- **Resultado general:** APROBADO
- **Notebook de evidencia:** `notebooks/05_validacion_final.ipynb`

## Conciliación de registros

| Indicador | Raw | Bronze | Silver | Gold |
| :--- | ---: | ---: | ---: | ---: |
| Pedidos únicos | 830 | 830 | 830 | 830 |
| Líneas de pedido | 2155 | 2155 | 2155 | 2155 |

No se detectaron registros perdidos ni duplicados durante el procesamiento.

## Conciliación financiera

| Métrica | Raw | Bronze | Silver | Gold | Diferencia Raw-Gold |
| :--- | ---: | ---: | ---: | ---: | ---: |
| Venta bruta | $1,354,458.59 | $1,354,458.59 | $1,354,458.59 | $1,354,458.59 | $0.00 |
| Descuentos | $88,665.57 | $88,665.57 | $88,665.57 | $88,665.57 | $0.00 |
| Venta neta | $1,265,793.02 | $1,265,793.02 | $1,265,793.02 | $1,265,793.02 | $0.00 |

## Controles ejecutados

{tabla_controles}

## Validación dimensional

- `Dim_Calendario`: 1096 claves únicas.
- `Dim_Cliente`: 91 claves únicas.
- `Dim_Producto`: 77 claves únicas.
- `Dim_Empleado`: 9 claves únicas.
- `Dim_Transportista`: 3 claves únicas.
- Claves primarias duplicadas: 0.
- Claves primarias nulas: 0.
- Claves foráneas huérfanas en `Fact_Ventas`: 0.

## Cobertura temporal

- Rango de `Dim_Calendario`: 1996-01-01 a 1998-12-31.
- Rango de ventas: 1996-07-04 a 1998-05-06.
- Días en calendario: 1096.
- Saltos en la secuencia diaria: 0.
- Ventas fuera del calendario: 0.

## Conclusión

El pipeline superó satisfactoriamente todos los controles de calidad y
conciliación. La capa Gold representa fielmente los datos de origen, con una
diferencia financiera de $0.00 entre Raw y Gold.

**Estado final: APROBADO PARA INTEGRACIÓN CON POWER BI.**
"""

ruta_acta.write_text(contenido_acta, encoding="utf-8")

if not ruta_acta.exists() or ruta_acta.stat().st_size == 0:
    raise IOError("No fue posible generar el acta de validación.")

print("=== GENERACIÓN DEL ACTA TÉCNICA ===")
print()
print("[OK] Acta generada correctamente.")
print(f"Archivo: {ruta_acta.resolve()}")
print(f"Tamaño: {ruta_acta.stat().st_size:,} bytes")

=== GENERACIÓN DEL ACTA TÉCNICA ===

[OK] Acta generada correctamente.
Archivo: docs\acta_validacion_pipeline.md
Tamaño: 2,280 bytes
